In [61]:
import pandas as pd
import numpy as np

# 엑셀 불러오기
df = pd.read_excel("./data/구직사이트 공고(260316).xlsx", sheet_name="마감공고(DB용)", header=2)

In [62]:
df.head()

,기업명,지원마감,지역,고용형태,학력,직무,스킬,우대사항,급여,지원방법,선택
0,페픽,2024-01-01 00:00:00,광주,정규직,학력무관,웹퍼빌리싱,"웹디자인, HTML, CSS, JS",- 해당 직무 인턴 경력자\n- 인근거주자\n- 운전가능자,회사내규,https://www.saramin.co.kr/zf_user/jobs/relay/v...,False
1,링크캠퍼스,2024-01-02 00:00:00,광주,정규직,학력무관,"웹,앱개발","Js, MySQL, PHP, React, DBMS",- 해당 직무 인턴 경력자\n- 인근거주자\n- 운전가능자,"연봉 2,413 ~",https://www.jobkorea.co.kr/Recruit/GI_Read/436...,False
2,스마트컴퍼니,2024-01-03 00:00:00,광주,정규직,초대졸이상,웹퍼빌리싱,"홈페이지 디자인, 퍼블리싱, 페이지제작",- 해당 직무 인턴 경력자\n- 인근거주자\n- 운전가능자,회사내규,https://www.saramin.co.kr/zf_user/jobs/relay/v...,False
3,온메이커스,2024-01-04 00:00:00,대전,정규직,학력무관,웹기획,"웹 서비스 기획, 요구사항 분석",- 해당 직무 인턴 경력자\n- 인근거주자\n- 운전가능자,회사내규,https://www.jobkorea.co.kr/Recruit/GI_Read/436...,False
4,(주)케이티지,2024-01-07 00:00:00,대전,정규직,대졸이상,웹개발,"Java, JSP, JS",- 관련업무 경험자 우대,회사내규,https://www.jobkorea.co.kr/Recruit/GI_Read/435...,False


### 데이터 전처리
- 분석에 필요하지 않은 '선택' 컬럼 제거
- 결측치가 너무 많아 사용하기 어려운 '우대사항' 컬럼 제거
- 공고등록일 정보가 없으므로, 일괄적으로 '지원마감' 컬럼의 날짜형에서 1달을 빼서 추정 컬럼으로 추가
- 상시채용/채용시마감의 경우에는 공고등록일 추정이 불가능하고, 총 데이터의 수(약 2400)에 비해서 적은 양(약 100)이기 때문에 분석에서 제외

In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2776 entries, 0 to 2775
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   기업명     2409 non-null   object
 1   지원마감    2409 non-null   object
 2   지역      2409 non-null   object
 3   고용형태    2409 non-null   object
 4   학력      2409 non-null   object
 5   직무      2407 non-null   object
 6   스킬      2408 non-null   object
 7   우대사항    114 non-null    object
 8   급여      2409 non-null   object
 9   지원방법    2409 non-null   object
 10  선택      2776 non-null   bool  
dtypes: bool(1), object(10)
memory usage: 219.7+ KB


In [64]:
# 분석에 의미가 없는 선택 컬럼 제거
df.drop(columns=['선택'], inplace=True)

In [65]:
# 결측치가 너무 많아 사용하기 어려운 우대사항 컬럼 제거
df.drop(columns=['우대사항'], inplace=True)

In [66]:
# NaN으로만 이루어진 행(선택 컬럼 때문에 발생)을 제거
df = df.dropna(axis=0, how='all')

In [67]:
# 결측치 수 확인
df.isna().sum()

기업명     0
지원마감    0
지역      0
고용형태    0
학력      0
직무      2
스킬      1
급여      0
지원방법    0
dtype: int64

In [68]:
# 스킬 컬럼에, 빈 값은 아니지만 문자열 '-'로 들어있어 사실상 비어있는 것이나 다름없는 행의 수 확인
(df['스킬'].str.strip() == '-').sum()

np.int64(311)

In [69]:
df = df.replace(r'^\s*$', np.nan, regex=True)

# 직무에 결측치가 존재하는 행 제거
df = df.dropna(subset=['직무'])

# 스킬 컬럼의 결측치와 '-'로 채워져 있는 행 제거
df = df[
    df['스킬'].notna() &
    (df['스킬'].str.strip() != '-')
].copy()

In [70]:
# 지원마감 컬럼에 데이터가 상시채용/채용시마감으로 들어있는 행 제거
df = df[~df['지원마감'].astype(str).str.strip().isin(['상시채용', '상시 채용', '채용시마감', '채용시 마감'])].copy()

In [71]:
df['지원마감'].astype(str).str.strip().isin(['상시채용', '상시 채용', '채용시마감', '채용시 마감']).sum()

np.int64(0)

In [72]:
df

,기업명,지원마감,지역,고용형태,학력,직무,스킬,급여,지원방법
0,페픽,2024-01-01 00:00:00,광주,정규직,학력무관,웹퍼빌리싱,"웹디자인, HTML, CSS, JS",회사내규,https://www.saramin.co.kr/zf_user/jobs/relay/v...
1,링크캠퍼스,2024-01-02 00:00:00,광주,정규직,학력무관,"웹,앱개발","Js, MySQL, PHP, React, DBMS","연봉 2,413 ~",https://www.jobkorea.co.kr/Recruit/GI_Read/436...
2,스마트컴퍼니,2024-01-03 00:00:00,광주,정규직,초대졸이상,웹퍼빌리싱,"홈페이지 디자인, 퍼블리싱, 페이지제작",회사내규,https://www.saramin.co.kr/zf_user/jobs/relay/v...
3,온메이커스,2024-01-04 00:00:00,대전,정규직,학력무관,웹기획,"웹 서비스 기획, 요구사항 분석",회사내규,https://www.jobkorea.co.kr/Recruit/GI_Read/436...
4,(주)케이티지,2024-01-07 00:00:00,대전,정규직,대졸이상,웹개발,"Java, JSP, JS",회사내규,https://www.jobkorea.co.kr/Recruit/GI_Read/435...
...,...,...,...,...,...,...,...,...,...
2403,펑타이그레이터차이나(유),2026.01.30,서울 강남구,인턴직,대졸(4년제) 이상,AI 리서치,Python,면접 후 결정,https://www.saramin.co.kr/zf_user/jobs/relay/v...
2405,(주)메타엠,2026.01.30,서울 중구,정규직,학력무관,AI 엔지니어,"Linux, Python, 머신러닝, 딥러닝",면접 후 결정,https://www.saramin.co.kr/zf_user/jobs/relay/v...
2406,(주)스타일셀러,2026.01.30,서울 서초구,정규직\n계약직,학력무관,AI 활용 및 자동화,"Python, JavaScript, API",면접 후 결정,https://www.saramin.co.kr/zf_user/jobs/relay/v...
2407,(주)지엠피아이티,2026.01.30,서울 금천구,정규직,학력무관,"프론트엔드, 백엔드",React,면접 후 결정,https://www.saramin.co.kr/zf_user/jobs/relay/v...


In [73]:
# 지원마감 컬럼의 데이터 형태 통일
# 앞 10개 문자까지만 추출하여 날짜 데이터로 사용
df['지원마감'] = (
    df['지원마감']
    .astype(str)
    .str.strip()
    .str[:10]
    .str.replace('.', '-', regex=False)
)

df['지원마감'] = pd.to_datetime(df['지원마감'], errors='coerce')

In [74]:
df['지원마감']

0      2024-01-01
1      2024-01-02
2      2024-01-03
3      2024-01-04
4      2024-01-07
          ...    
2403   2026-01-30
2405   2026-01-30
2406   2026-01-30
2407   2026-01-30
2408   2026-01-30
Name: 지원마감, Length: 2028, dtype: datetime64[ns]

In [75]:
print(df['지원마감'].dtype)
print(df['지원마감'].isna().sum())

datetime64[ns]
0


In [76]:
# 2025년 이전의 데이터 제거
df = df[df['지원마감'].dt.year >= 2025].copy()

In [77]:
df['지원마감']

1077   2025-01-05
1078   2025-01-04
1126   2025-01-04
1177   2025-01-01
1178   2025-01-01
          ...    
2403   2026-01-30
2405   2026-01-30
2406   2026-01-30
2407   2026-01-30
2408   2026-01-30
Name: 지원마감, Length: 992, dtype: datetime64[ns]

In [79]:
# 지원마감일에서 30일 이전을 공고등록일로 추정하여 컬럼 생성
df['공고등록일'] = df['지원마감'] - pd.Timedelta(days=30)

In [81]:
# 공고등록일이 2025년 이전인 행 제거
df = df[df['공고등록일'] >= '2025-01-01'].copy()

In [83]:
# 공고등록일 컬럼을 지원마감 컬럼 왼쪽으로 당겨오기

cols = df.columns.tolist()

cols.remove('공고등록일')
idx = cols.index('지원마감')
cols.insert(idx, '공고등록일')

df = df[cols]

In [85]:
# 날짜순 정렬
df = df.sort_values(by='공고등록일', ascending=True).copy()

In [87]:
# 인덱스 정렬
df = df.sort_values(by='공고등록일', ascending=True).reset_index(drop=True)

In [88]:
df

,기업명,공고등록일,지원마감,지역,고용형태,학력,직무,스킬,급여,지원방법
0,타이아,2025-01-01,2025-01-31,광주,정규직,대졸이상,응용S/W 개발자,"Android, JAVA, Python, MySQL, AI, 머신러닝, 딥러닝","연봉 3,100만원 이상",https://www.jobkorea.co.kr/Recruit/GI_Read/461...
1,타이아,2025-01-01,2025-01-31,광주,정규직,대졸이상,"웹, 앱 개발자","HTML5, CSS, JAVA, JSP, Spring","연봉 3,100만원 이상",https://www.jobkorea.co.kr/Recruit/GI_Read/461...
2,타이아,2025-01-01,2025-01-31,광주,정규직,대졸이상,시스템팀,클라우드,"연봉 3,100만원 이상",https://www.jobkorea.co.kr/Recruit/GI_Read/461...
3,㈜이글루코퍼레이션,2025-01-01,2025-01-31,광주,정규직,초대졸이상,프론트엔드,"CSS, Django, Docker, Spring Boot, Spring Frame...",회사내규에 따름,https://www.jobkorea.co.kr/Recruit/GI_Read/462...
4,㈜에프앤가이드,2025-01-01,2025-01-31,서울시 강서구,정규직,대졸이상,개발자,"SQL, Python",회사내규에 따름,https://www.jobkorea.co.kr/Recruit/GI_Read/462...
...,...,...,...,...,...,...,...,...,...,...
879,㈜인테크,2026-01-30,2026-03-01,광주,정규직,대졸이상,클라우드 엔지니어,"Docker, kubernetes","연봉 3,000~3,400만원",https://www.jobkorea.co.kr/Recruit/GI_Read/483...
880,㈜미르이즈,2026-02-08,2026-03-10,광주,정규직,대졸이상,AI 응용프로그램 개발자,"JAVA, Python, Pytorch, Tensorflow, LLM",회사 내규에 따름,https://www.jobkorea.co.kr/Recruit/GI_Read/483...
881,주식회사 이지커넥트,2026-02-08,2026-03-10,전남 여수시,정규직,학력무관,웹 개발자,"JAVA, Spring, RDBMS, HTML, JavaScript",면접 후 결정,https://www.saramin.co.kr/zf_user/jobs/relay/v...
882,한솔,2026-02-10,2026-03-12,전남 순천시,정규직\n계약직\n프리랜서,대졸(4년제) 이상,"백엔드, 프론트엔드","Python, Node.js, AWS, React, AI, ML",면접 후 결정,https://www.saramin.co.kr/zf_user/jobs/relay/v...


In [89]:
df.to_excel('전처리완료_공고데이터.xlsx', index=False)